# LangChain — the original guide (Groq + Gemini)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/00_langchain_original/00_langchain_original.ipynb)

**Author:** Dr. Nasser Mooman · Magan AI Inc. · nmooman@gmail.com · written with help from Claude (Anthropic)

> ⚠️ **Teaching code — do not use in production.** Short on purpose: no authentication, input validation, prompt-injection protection, error handling or tests. LLM answers can be wrong. Provided as is, without warranty.

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

The original *How to Build AI Agents Completely Free* code, unchanged: `create_agent` with a free Groq model, tools, memory and a Groq → Gemini fallback. Every other folder rebuilds **this same agent** in a different framework.

Needs `GROQ_API_KEY` (and `GOOGLE_API_KEY` for the fallback).

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "langchain" "langchain-groq" "langchain-google-genai" "langchain-community" "duckduckgo-search" "python-dotenv"

### API keys
**Bring your own LLM API key** (Groq and Gemini have free tiers). No key is needed only if you already run a local server (Ollama, SGLang or vLLM): set `LLM_PROVIDER=ollama`, `OLLAMA_BASE_URL` and `LLM_MODEL`.

* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ["LLM_PROVIDER"] = "groq"             # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

## 1. `01_first_agent.py`

guide section — "your first agent"
a complete, working agent: free groq model + a system prompt, no tools yet.

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq

# load your GROQ_API_KEY from the .env file
load_dotenv()

# the brain: a free groq model
model = ChatGroq(model="llama-3.3-70b-versatile")

# the agent: model + a system prompt telling it how to behave
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)

# ask it something
result = agent.invoke(
    {"messages": [{"role": "user", "content": "explain what an AI agent is in two sentences."}]}
)

# print the agent's reply
print(result["messages"][-1].content)

## 2. `02_agent_with_tools.py`

guide section — "giving your agent tools"
adds two tools: a prebuilt web search + a function you wrote yourself.

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

_ddg = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en", time=None))


# tool 1 — web search (prebuilt, free, no key)
# WORKSHOP FIX (the only change vs. the original repo): the free search is flaky — rate limits and
# random backend errors. we wrap it so a failed search is reported to the agent instead of crashing.
@tool
def search(query: str) -> str:
    """Search the web (DuckDuckGo) for up-to-date information."""
    try:
        return _ddg.invoke(query)
    except Exception as e:
        return f"search failed ({e}). try again, or answer from what you know."


# tool 2 — a function you wrote yourself
@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


# hand both tools to the agent
agent = create_agent(
    model=model,
    tools=[search, word_count],
    system_prompt="You are a helpful assistant. Use your tools when they help answer accurately.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "search for the latest langchain version, then tell me how many words your answer is."}]}
)

print(result["messages"][-1].content)

## 3. `03_agent_with_memory.py`

guide section — "giving your agent memory"
a checkpointer saves the conversation; a thread_id ties messages into one chat.

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

# the checkpointer saves conversation state between calls
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful assistant. Be concise and accurate.",
    checkpointer=InMemorySaver(),
)

# a thread_id labels the conversation — same id = same memory
config = {"configurable": {"thread_id": "chat-1"}}

# first message
r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "hi! my name is m0h."}]},
    config,
)
print(r1["messages"][-1].content)

# second message — same thread_id, so it remembers
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "what's my name?"}]},
    config,
)
print(r2["messages"][-1].content)

## 4. `04_agent_with_fallback.py`

guide section — "the fallback setup"
try groq first; if it fails (rate limit / removed model / outage), fall back to gemini.

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


# define your providers in order of preference — all free
def get_model():
    try:
        # first choice: groq (fastest)
        model = ChatGroq(model="llama-3.3-70b-versatile")
        model.invoke("ping")  # quick test that it actually responds
        print("using groq")
        return model
    except Exception as e:
        # groq is down or rate-limited — fall back to gemini
        print(f"groq failed ({e}); falling back to gemini")
        return ChatGoogleGenerativeAI(model="gemini-2.5-flash")


agent = create_agent(
    model=get_model(),
    tools=[],
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "say hi and tell me which model you are."}]}
)
print(result["messages"][-1].content)

## 5. `agent.py`

guide — the complete agent (bonus: all three upgrades combined)
provider fallback (groq -> gemini) + tools (search + word_count) + memory.

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langgraph.checkpoint.memory import InMemorySaver

# load GROQ_API_KEY and GOOGLE_API_KEY from .env
load_dotenv()


# --- fallback: try groq first, fall back to gemini (both free) ---
def get_model():
    try:
        model = ChatGroq(model="llama-3.3-70b-versatile")
        model.invoke("ping")  # quick test that it actually responds
        print("using groq")
        return model
    except Exception as e:
        print(f"groq failed ({e}); falling back to gemini")
        return ChatGoogleGenerativeAI(model="gemini-2.5-flash")


# --- tools: a prebuilt web search + a function you wrote yourself ---
_ddg = DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en", time=None))


# WORKSHOP FIX (the only change vs. the original repo): the free search is flaky — rate limits and
# random backend errors. we wrap it so a failed search is reported to the agent instead of crashing.
@tool
def search(query: str) -> str:
    """Search the web (DuckDuckGo) for up-to-date information."""
    try:
        return _ddg.invoke(query)
    except Exception as e:
        return f"search failed ({e}). try again, or answer from what you know."


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


# --- memory: InMemorySaver keeps conversation state in RAM across calls ---
agent = create_agent(
    model=get_model(),
    tools=[search, word_count],
    system_prompt=(
        "You are a helpful assistant. Use your tools when they help answer "
        "accurately, and remember the conversation."
    ),
    checkpointer=InMemorySaver(),
)


def main():
    # same thread_id = same conversation
    config = {"configurable": {"thread_id": "chat-1"}}

    # turn 1 — give it something to remember + a task that needs tools
    r1 = agent.invoke(
        {"messages": [{"role": "user", "content": "my name is m0h. search for the latest langchain version, then tell me how many words your answer is."}]},
        config,
    )
    print(r1["messages"][-1].content)

    # turn 2 — same thread, so it should recall the name
    r2 = agent.invoke(
        {"messages": [{"role": "user", "content": "what's my name?"}]},
        config,
    )
    print(r2["messages"][-1].content)


if __name__ == "__main__":
    main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*